<a href="https://colab.research.google.com/github/MLDreamer/Linkedin-posts/blob/main/LLM_as_Fancy_Markov_chains%3F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from matplotlib.animation import FuncAnimation
import networkx as nx
from PIL import Image, ImageDraw, ImageFont
import io

def create_llm_markov_animation():
    fig, ax = plt.subplots(figsize=(12, 8))

    frames = []

    vocab = ['0', '1']
    context_window = 3

    all_states = []
    for length in range(1, context_window + 1):
        for i in range(2**length):
            state = format(i, f'0{length}b')
            all_states.append(state)

    def get_next_token(state):
        if len(state) >= 3:
            sum_bits = sum(int(bit) for bit in state[-3:])
        else:
            sum_bits = sum(int(bit) for bit in state)
        return '0' if sum_bits % 2 == 0 else '1'

    def get_next_state(current_state, next_token):
        if len(current_state) >= context_window:
            return current_state[1:] + next_token
        else:
            return current_state + next_token

    def create_frame(step, current_state=None, next_token=None, next_state=None,
                    show_rule=False, show_states=False, show_transitions=False):
        ax.clear()
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 8)
        ax.axis('off')

        if step == 0:
            ax.text(5, 7, 'Large Language Models as Markov Chains',
                   fontsize=20, ha='center', weight='bold')
            ax.text(5, 6, 'A Simple Example with Binary Tokens',
                   fontsize=14, ha='center')
            ax.text(5, 5, 'Vocabulary: {0, 1}', fontsize=12, ha='center')
            ax.text(5, 4.5, 'Context Window: 3 tokens', fontsize=12, ha='center')

        elif step == 1:
            ax.text(5, 7.5, 'Step 1: Define the Rule', fontsize=16, ha='center', weight='bold')
            ax.text(5, 6.5, 'Our toy LLM follows a simple pattern:', fontsize=12, ha='center')
            ax.text(5, 6, 'If sum of last 3 bits is EVEN → predict 0', fontsize=12, ha='center')
            ax.text(5, 5.5, 'If sum of last 3 bits is ODD → predict 1', fontsize=12, ha='center')

            ax.text(2, 4, 'Example:', fontsize=12, weight='bold')
            ax.text(2, 3.5, 'State: "110"', fontsize=11)
            ax.text(2, 3, 'Sum: 1+1+0 = 2 (even)', fontsize=11)
            ax.text(2, 2.5, 'Next token: 0', fontsize=11, color='red')

        elif step == 2:
            ax.text(5, 7.5, 'Step 2: States as Token Sequences', fontsize=16, ha='center', weight='bold')
            ax.text(5, 6.5, 'Each possible sequence is a STATE in our Markov chain', fontsize=12, ha='center')

            states_1 = ['0', '1']
            states_2 = ['00', '01', '10', '11']
            states_3 = ['000', '001', '010', '011', '100', '101', '110', '111']

            ax.text(2, 5.5, 'Length 1:', fontsize=11, weight='bold')
            ax.text(3, 5.5, ', '.join(states_1), fontsize=11)

            ax.text(2, 5, 'Length 2:', fontsize=11, weight='bold')
            ax.text(3, 5, ', '.join(states_2), fontsize=11)

            ax.text(2, 4.5, 'Length 3:', fontsize=11, weight='bold')
            ax.text(3, 4.5, ', '.join(states_3), fontsize=11)

            ax.text(5, 3.5, f'Total States: {len(all_states)}', fontsize=12, ha='center', weight='bold')

        elif step >= 3 and step <= 10:
            demo_states = ['0', '01', '011', '110', '100', '001', '010', '101']
            current_idx = step - 3

            if current_idx < len(demo_states):
                current = demo_states[current_idx]
                next_tok = get_next_token(current)
                next_st = get_next_state(current, next_tok)

                ax.text(5, 7.5, f'Step {step}: State Transition Example', fontsize=16, ha='center', weight='bold')

                state_rect = patches.Rectangle((2, 5.5), 2, 1, linewidth=2, edgecolor='blue', facecolor='lightblue')
                ax.add_patch(state_rect)
                ax.text(3, 6, f'Current State: "{current}"', fontsize=12, ha='center', weight='bold')

                if len(current) >= 3:
                    bits = current[-3:]
                    sum_bits = sum(int(b) for b in bits)
                else:
                    bits = current
                    sum_bits = sum(int(b) for b in bits)

                ax.text(3, 4.5, f'Last {len(bits)} bits: {bits}', fontsize=11, ha='center')
                ax.text(3, 4, f'Sum: {sum_bits} ({"even" if sum_bits % 2 == 0 else "odd"})', fontsize=11, ha='center')

                arrow = patches.FancyArrowPatch((4.2, 6), (5.8, 6),
                                              arrowstyle='->', mutation_scale=20, color='red')
                ax.add_patch(arrow)
                ax.text(5, 6.3, f'Predict: {next_tok}', fontsize=11, ha='center', color='red', weight='bold')

                next_rect = patches.Rectangle((6, 5.5), 2, 1, linewidth=2, edgecolor='green', facecolor='lightgreen')
                ax.add_patch(next_rect)
                ax.text(7, 6, f'Next State: "{next_st}"', fontsize=12, ha='center', weight='bold')

        elif step == 11:
            ax.text(5, 7.5, 'Step 11: Building the Markov Chain', fontsize=16, ha='center', weight='bold')
            ax.text(5, 6.8, 'Each state connects to its possible next states', fontsize=12, ha='center')

            G = nx.DiGraph()
            sample_states = ['00', '01', '10', '11']

            pos = {
                '00': (2, 5),
                '01': (2, 3),
                '10': (8, 5),
                '11': (8, 3)
            }

            for state in sample_states:
                G.add_node(state)
                next_token = get_next_token(state)
                next_state = get_next_state(state, next_token)
                if next_state in sample_states:
                    G.add_edge(state, next_state)

            nx.draw(G, pos, ax=ax, with_labels=True, node_color='lightblue',
                   node_size=1500, font_size=12, font_weight='bold', arrows=True)

            ax.text(5, 2, 'This is a Markov Chain!', fontsize=14, ha='center', weight='bold', color='red')

        elif step == 12:
            ax.text(5, 7.5, 'Step 12: Why This Matters for LLMs', fontsize=16, ha='center', weight='bold')

            points = [
                '• LLMs predict next tokens like Markov chains',
                '• Context window = finite memory',
                '• Vocabulary size = finite states',
                '• Training learns transition probabilities',
                '• Theory helps understand generalization'
            ]

            for i, point in enumerate(points):
                ax.text(1, 6 - i*0.8, point, fontsize=12, va='center')

        elif step == 13:
            ax.text(5, 7, 'Real LLMs: Same Principle, Bigger Scale', fontsize=16, ha='center', weight='bold')
            ax.text(5, 6, 'GPT-4: ~50K vocabulary, 32K context window', fontsize=12, ha='center')
            ax.text(5, 5.5, 'States ≈ 50K^32K (astronomically large!)', fontsize=12, ha='center')
            ax.text(5, 5, 'But same mathematical structure', fontsize=12, ha='center')

            ax.text(5, 3.5, 'This framework enables:', fontsize=12, ha='center', weight='bold')
            ax.text(5, 3, '• Theoretical analysis of LLM behavior', fontsize=11, ha='center')
            ax.text(5, 2.5, '• Better understanding of in-context learning', fontsize=11, ha='center')
            ax.text(5, 2, '• Improved model design and training', fontsize=11, ha='center')

    for step in range(14):
        create_frame(step)

        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', dpi=100)
        buf.seek(0)
        img = Image.open(buf).copy()
        frames.append(img)
        buf.close()

    duration_per_frame = 3000
    frames[0].save(
        'llm_markov_chain.gif',
        save_all=True,
        append_images=frames[1:],
        duration=duration_per_frame,
        loop=0
    )

    plt.close()
    print("Animation saved as 'llm_markov_chain.gif'")

    from google.colab import files
    files.download('llm_markov_chain.gif')

create_llm_markov_animation()

Animation saved as 'llm_markov_chain.gif'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>